# Studio 8 -- The Three Body Problem
## Put your group number and the full names of you and your groupmates here:
### Group: #
- 1 -- 
- 2 -- 
- 3 --



In [ ]:
## run this cell first to import all the necessary tools
import numpy as np
from matplotlib import pyplot as plt 
import matplotlib

matplotlib.rcParams['animation.embed_limit'] = 2**128
from scipy import integrate
import matplotlib.animation as animation
from IPython.display import HTML

The [three-body problem](https://en.wikipedia.org/wiki/Three-body_problem), the motion of three particles under their mutual gravitational force, is a centuries old problem in physics going back to Isaac Newton.  There are many different regimes of the three-body problem that depend on the masses and seperations of the particles.  Some of these are stable (like the Earth-Moon-Sun system), but most configurations are not stable in general.  If I start with three particles at rest, almost any generic configuration will be unstable.

# Problem 1

## Setting up and testing right-hand side of the ODE

In Homework 8, you worked out the 18-element vector right-hand side for the three-body problem, expressed as a first-order ODE:

$$ \frac{d\vec{f}}{dt} = \vec{F}(\vec{f})$$

## Part A)

**Comparing with your group-mates, implement the right-hand side of Newton's second law for gravity in the the function below (part B from Homework 08)**.  You may find it useful to use intermediate variables such as the seperations between particles

Like good theorists, we will assume all the masses and $G$ are equal to 1.

**NOTE!!:** A couple of the later functions require a specific order for your vector.  I have provided it in the function below.  Don't mess with the order!

In [ ]:
def right_hand_side(t,vec):
    
    '''
    Returns the right-hand side of the 1st-order form of Newton's 2nd Law (above equation)
        
    PARAMETERS:
        t   - time (not needed for our problem)
        vec - input state vector (\vec{f} from above)
    
    RETURN VALUE:
        vecout - 18 Element NumPy array (\vec{F} from above)
    '''
    
    G=1 

    x1,y1,z1,vx1,vy1,vz1,x2,y2,z2,vx2,vy2,vz2,x3,y3,z3,vx3,vy3,vz3 = vec
    
    ## Put your solution from HW08 here
    fx1 = vx1 ## and so on
    
    vecout = fx1,fy1,fz1,fvx1,fvy1,fvz1,fx2,fy2,fz2,fvx2,fvy2,fvz2,fx3,fy3,fz3,fvx3,fvy3,fvz3 

    return vecout


## Part B)

There are a **handful of stable configurations** of the three-body problem.  I've included the initial conditions for one of them (the figure-8) below.

**Using the** [solve_ivp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html#scipy.integrate.solve_ivp) **function** from the `scipy.integrate` package, integrate your ODE forward 25 dynamical times with the provided initial conditions.  You will want to have the output save the state of the system 10 times per dynamical time (so 250 time samples total).  This is what `solve_ivp` function calls `t_eval`.  I have provided the code to do this below.

**Plot $x(t)$ vs $y(t)$** for all three particles.  Does this seem to be stable?

**Hint:** For better convergence, use the `DOP853` method of integration, and set `rtol` (the accuracy parameter) to a small value (e.g. $10^{-8}$)

In [ ]:
## These are the initial conditions for one of the few *stable* three-body problems
initial_conditions_stable = np.array([0.97000436,-0.24308753,0,0.93240737/2,0.86473146/2,0,
                               -0.97000436,0.24308753,0,0.93240737/2,0.86473146/2,0,
                               0,0,0,-0.93240737,-0.86473146,0])

times = np.linspace(0,25,250)

## Integrate your initial value problem here
output = integrate.solve_ivp(right_hand_side,
                             (0,250),
                             initial_conditions_stable,
                             t_eval=times,
                             method='DOP853',
                            rtol=1e-8)

print(output.message)

## The data (an 18x250 NumPy array) is saved in output.y.  Unpack it here
x1,y1,z1,vx1,vy1,vz1,x2,y2,z2,vx2,vy2,vz2,x3,y3,z3,vx3,vy3,vz3 = output.y

In [ ]:
## Put Plot here

## Part C) Create Aniomation

The function below is designed to create an in-line animation of your particles over time.  Take a look at the documentation to see what it's doing, then use the command below it to create a time-dependent animation of your three-body integration.


In [ ]:
def animate_three_body_problem(output_vector,fps=30,boxsize=1.2,N_skip=1):
    
    '''
    Creates an Animation of the motion of three particles
        
    PARAMETERS:
        output_vector -- the 18-element vector of your integrated 3-body problem
        fps           -- Frames per second of the animation (default = 30)
        boxsize       -- How big to make the animation (default = 1.2)
        N_skip        -- How many frames to skip (default = 1)
    
    RETURN VALUE:
        anim -- Matplotlib animation 
    '''
    
    x1,y1,z1,vx1,vy1,vz1,x2,y2,z2,vx2,vy2,vz2,x3,y3,z3,vx3,vy3,vz3 = output_vector
    

    fig,ax = plt.subplots()

    ## Convert from FPS to interval (milliseconds between frames)
    interval = 1000//fps
    N_total = len(x1)
    N_frames = N_total//N_skip
    
    xs = [x1,x2,x3]
    ys = [y1,y2,y3]

    scatter_plots = []
    line_plots = []
    
    ## Create the initial scatter plot and line plot
    for i in range(3):
        scatter_plots.append(ax.scatter(xs[i][0], ys[i][0],edgecolors='black'))
        line_plots.append(ax.plot(xs[i][0],ys[i][0],alpha=0.8,zorder=-1)[0])

    ## Definte the function we'll call to generate the plots
    def update(frame):
        for i in range(3):
            ## Update the scatter plots
            scatter_plots[i].set_offsets([xs[i][frame*N_skip],ys[i][frame*N_skip]])

            ## And the line plots
            line_plots[i].set_xdata(xs[i][:frame*N_skip])
            line_plots[i].set_ydata(ys[i][:frame*N_skip])

        ## Also reset the size of the axes
        ax.set_xlim(-boxsize,boxsize)
        ax.set_ylim(-boxsize,boxsize)

    ## The animation 
    ani = animation.FuncAnimation(fig=fig, 
                                  func=update, 
                                  frames=N_frames, 
                                  interval=interval)
    
    plt.close()
    
    return ani

In [ ]:
anim = animate_three_body_problem(output.y)
HTML(anim.to_jshtml())

# Problem 2

## Unstable configurations and chaos theory

Formally speaking, most configurations of the three-body problem you can cook up are unstable, and **chaotic**, meaning that **slight changes in initial conditions** produce completely different outputs.


## Part A)

Using the initial conditions below of three bodies initally at rest **integrate this unstable fewbody problem for 1000 dynamical times**.  Use the same integration call from above, but this time **integrate for 1000 dynamical times**.  Save 3000 time evaluations

**Plot $x(t)$ and $y(t)$ for the three particles**.  What is happening?

In [ ]:
initial_conditions_unstable = np.array([-2,-2,0,0,0,0,2,-1,0,0,0,0,0,1,0,0,0,0])

In [ ]:
## Put your integration using solve_ivp here.  
## You can just copy the cell above (but with some appropriate changes)

In [ ]:
## Put your plot of the positions of the three particles here

## Part B)

Try tweaking the initial conditions above just a little bit (but keep your particles in the X-Y plane).  Make a new plot of the outcome.  Does the outcome seem similar to the previous system?

In [ ]:
## Put your new integration with different initial conditions here

In [ ]:
## Put your new plot here

## Part C)

Make an animation of one of your three-body integrations.  Use the `boxsize` parameter to shrink your plot to show the interesting parts of the integration.

**Hint:** You may want to set `N_skip=5`, which will only include every 5th integration in the animation.  Otherwise it will take a while...

In [ ]:
## Put your call to the animation function here